In [1]:
# -------- Config --------
DATADIR = "/work/gr-fe/bryan/data/TCGA"
COHORT = "TCGA-BRCA"
META_PICKLE = f"{DATADIR}/{COHORT}/02_processed/phenotype.cleaned.pkl"
OMICS = ["mRNA", "miRNA", "DNAm", "CNV", "RPPA"]  # datExpr_{omic}.csv for each
RAW_OMICS_DIR = f"{DATADIR}/{COHORT}/01_raw/OMICS"
OUT_DIR = f"{DATADIR}/{COHORT}/02_processed"
OVERWRITE = True  # overwrite existing output pickles

# -------- Imports --------
import os
import pickle
from pathlib import Path
import pandas as pd

# -------- Helpers --------
def load_meta_ids(meta_path: str) -> pd.Series:
    """
    Load a pickle expected to contain either:
      - a pandas DataFrame with column 'ID', or
      - a dict with key 'ID' (array-like)
    Returns a clean string Series of IDs (duplicates removed, NaNs dropped).
    """
    # Try pandas first (faster for DataFrame pickles), fall back to pickle.load
    meta = None
    try:
        meta = pd.read_pickle(meta_path)
    except Exception:
        with open(meta_path, "rb") as f:
            meta = pickle.load(f)

    if isinstance(meta, pd.DataFrame) and "ID" in meta.columns:
        ids = meta["ID"]
    elif isinstance(meta, dict) and "ID" in meta:
        ids = pd.Series(meta["ID"], name="ID")
    else:
        raise ValueError("Meta pickle must be a DataFrame with 'ID' column or a dict with key 'ID'.")

    ids = ids.dropna().astype(str)
    ids = ids[~ids.duplicated()].reset_index(drop=True)
    return ids

def orient_to_ids(df: pd.DataFrame, id_set: set) -> pd.DataFrame | None:
    """
    Ensure sample IDs are in the index; if they're in columns, transpose.
    If neither index nor columns contain any meta IDs, return None.
    """
    # Normalize types to str for safe matching
    df.index = df.index.map(str)
    df.columns = df.columns.map(str)

    idx_hit = len(id_set.intersection(df.index))
    col_hit = len(id_set.intersection(df.columns))

    if idx_hit == 0 and col_hit > 0:
        df = df.T
        idx_hit = len(id_set.intersection(df.index))

    if idx_hit == 0:
        return None
    return df

# -------- Run --------
out_dir = Path(OUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

ids = load_meta_ids(META_PICKLE)
id_set = set(ids)

print(f"Loaded {len(ids)} unique meta IDs from: {META_PICKLE}")
print(f"Output dir: {out_dir}\n")

results = {}
for omic in OMICS:
    src = Path(RAW_OMICS_DIR) / f"datExpr_{omic}.csv"
    if not src.exists():
        print(f"[{omic}] SKIP - not found: {src}")
        continue

    try:
        df = pd.read_csv(src, index_col=0, dtype=str)
    except Exception as e:
        print(f"[{omic}] ERROR reading {src}: {e}")
        continue

    df_oriented = orient_to_ids(df, id_set)
    if df_oriented is None:
        print(f"[{omic}] SKIP - no overlap between meta IDs and {src.name} (rows or columns).")
        continue

    # Preserve meta ID order in the subset
    keep_ids = [i for i in ids if i in df_oriented.index]
    sub = df_oriented.loc[keep_ids]

    out_path = out_dir / f"{omic}.pkl"
    if out_path.exists() and not OVERWRITE:
        print(f"[{omic}] Exists, not overwritten: {out_path} (shape={sub.shape})")
    else:
        with open(out_path, "wb") as f:
            pickle.dump({"expr": sub}, f, protocol=pickle.HIGHEST_PROTOCOL)
        print(f"[{omic}] Saved {sub.shape} to {out_path}")

    results[omic] = sub

# Optional: quick peek at one modality
for omic in OMICS:
    if omic in results:
        display(results[omic].head())
        break


Loaded 1096 unique meta IDs from: /work/gr-fe/bryan/data/TCGA/TCGA-BRCA/02_processed/phenotype.cleaned.pkl
Output dir: /work/gr-fe/bryan/data/TCGA/TCGA-BRCA/02_processed

[mRNA] Saved (1046, 29995) to /work/gr-fe/bryan/data/TCGA/TCGA-BRCA/02_processed/mRNA.pkl
[miRNA] Saved (1013, 1601) to /work/gr-fe/bryan/data/TCGA/TCGA-BRCA/02_processed/miRNA.pkl
[DNAm] Saved (778, 200000) to /work/gr-fe/bryan/data/TCGA/TCGA-BRCA/02_processed/DNAm.pkl
[CNV] Saved (1051, 60265) to /work/gr-fe/bryan/data/TCGA/TCGA-BRCA/02_processed/CNV.pkl
[RPPA] Saved (867, 464) to /work/gr-fe/bryan/data/TCGA/TCGA-BRCA/02_processed/RPPA.pkl


,ENSG00000000003.15,ENSG00000000005.6,ENSG00000000419.13,ENSG00000000457.14,ENSG00000000460.17,ENSG00000000938.13,ENSG00000000971.16,ENSG00000001036.14,ENSG00000001084.13,ENSG00000001167.14,...,ENSG00000288611.1,ENSG00000288612.1,ENSG00000288638.1,ENSG00000288648.1,ENSG00000288657.1,ENSG00000288658.1,ENSG00000288663.1,ENSG00000288670.1,ENSG00000288674.1,ENSG00000288675.1
TCGA-E2-A1IU,9.95268139030024,4.09462213197204,10.9253024098081,10.8182852636041,8.89526477531777,8.63660208288473,10.5262135231799,10.9531619040005,12.0468507017367,11.4222629087557,...,3.64928268851503,5.06605351458121,2.84346716022229,2.84346716022229,2.84346716022229,3.64928268851503,4.71010720111342,8.12717534460742,3.82426747575019,4.20616911560394
TCGA-A1-A0SB,12.263477959988,9.50805076984095,10.4772148148614,10.2184990608179,8.68831239214972,7.18176658420779,13.3857434496814,9.88509887321799,10.7219741034037,11.4309675532413,...,2.84346716022229,5.4159895126938,2.84346716022229,2.84346716022229,2.84346716022229,6.7606038688169,5.3742590277243,8.08646230673897,4.66225334707816,5.45648051175065
TCGA-A2-A04W,12.1101859143097,3.52285660674024,11.606193795183,10.1188652598213,8.92496902972904,8.55992704748573,10.749512444256,11.8211892148652,10.1725494645339,11.0189141298844,...,2.84346716022229,5.34518305418988,4.31242202919176,7.77086336120383,2.84346716022229,4.75711344415342,4.75711344415342,7.71623298070183,3.52285660674024,6.10287173812148
TCGA-AN-A0AM,11.2169268215596,3.36439872656169,12.7847225805123,11.0767331353877,10.1392447197907,8.63085692457136,11.6295061532926,14.7623808980336,10.9908145381175,10.7576643003362,...,3.73633863789039,6.05652476237317,3.36439872656169,2.84346716022229,2.84346716022229,3.86925463732919,5.11375191713329,8.72420135706617,4.48980730845006,5.02869581857176
TCGA-LL-A440,11.1888015636889,6.72948123904209,10.381891924861,10.1329955496001,8.43991135974846,12.0983534873067,12.0665822887687,11.0765802333389,10.9608086749411,10.671238272171,...,3.52518569758943,6.60125046299762,2.84346716022229,2.84346716022229,3.52518569758943,4.44534503995852,6.12815216354478,8.15580974913908,5.16075915005576,5.66941937308753
